<a href="https://colab.research.google.com/github/kdang002/151B_SP26_Competition_forkbomb/blob/training_loop/starter_code_cse151b_comp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [4]:
##
# Install uv
#!wget -qO- https://astral.sh/uv/install.sh | sh

# Create a virtual environment
#!uv venv .venv --seed

# Install dependencies — this is fast thanks to uv's parallel resolver
#!.venv/bin/python -m pip install --upgrade pip
#!.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter accelerate peft

# Install Jupyter Kernel
#!.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

print("Done. Restart the kernel before proceeding.")
print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

Done. Restart the kernel before proceeding.
Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.


In [5]:
#google colab setup code
#!pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 accelerate

### Run the cell below every time to activate the installed environment.

In [6]:
# activate venv after installation. This needs to be run everytime.
#!source ./.venv/bin/activate

In [7]:
# Cell 1: Clone repo and install packages
!git clone https://github.com/kdang002/151B_SP26_Competition_forkbomb.git
import os
os.chdir('151B_SP26_Competition_forkbomb')

# Install dependencies
!pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 accelerate

fatal: destination path '151B_SP26_Competition_forkbomb' already exists and is not an empty directory.


In [8]:
# Cell 2: Setup configuration
import json
import os

MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"  # Change from "1" — Colab usually has GPU 0
DATA_PATH   = "/content/151B_SP26_Competition_forkbomb/data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 32768

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [9]:
#import json
#import os

# ── Configuration ─────────────────────────────────────────────────────────────
#MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
#GPU_ID      = "1"                    # CUDA_VISIBLE_DEVICES
#DATA_PATH   = "data/public.jsonl"
#OUTPUT_PATH = "results/starter_results.jsonl"
#MAX_TOKENS  = 32768

#os.environ["CUDA_VISIBLE_DEVICES"] = "0"

#import re
#import sys
#from pathlib import Path
#from typing import Optional

#from transformers import AutoTokenizer
#from vllm import LLM, SamplingParams
#from tqdm import tqdm

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [10]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


In [11]:
import random
from collections import defaultdict

SEED = 42
TEST_RATIO = 0.15
N_FOLDS = 5

rng = random.Random(SEED)
indexed_data = list(enumerate(data))
strata = defaultdict(list)
for idx, item in indexed_data:
    strata[int(bool(item.get("options")))].append(idx)

fold_assignments = {}
test_indices = []
trainval_indices = []
for bucket_indices in strata.values():
    rng.shuffle(bucket_indices)
    test_cut = max(1, int(round(len(bucket_indices) * TEST_RATIO)))
    bucket_test = bucket_indices[:test_cut]
    bucket_trainval = bucket_indices[test_cut:]
    test_indices.extend(bucket_test)
    trainval_indices.extend(bucket_trainval)
    for pos, idx in enumerate(bucket_trainval):
        fold_assignments[idx] = pos % N_FOLDS

def _select_records(indices):
    return [data[idx] for idx in indices]

test_data = _select_records(test_indices)
print(f"Total records: {len(data)} | Test records: {len(test_data)}")
print(f"Test split: {sum(bool(d.get('options')) for d in test_data)} MCQ, {sum(not d.get('options') for d in test_data)} free-form")
print(f"Train/Val to be split into {N_FOLDS} folds.")

Total records: 1126 | Test records: 169
Test split: 56 MCQ, 113 free-form
Train/Val to be split into 5 folds.


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [12]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Show concise, correct step-by-step reasoning. "
    "On the final line, output ONLY the final answer inside \\boxed{...}. "
    "If multiple values, separate them with commas (e.g. \\boxed{3, 7}). "
    "Also, display your chain-of-thoughts and show your work."
    "Do NOT output explanations, labels, or extra commentary."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. Read the question and the answer choices. "
    "Output ONLY the single uppercase letter of your chosen option inside \\boxed{...}, "
    "Also, display your chain-of-thoughts and show your work."
    "e.g. \\boxed{C}. No explanations, no extra text."
)

EXAMPLES = """Example 1 (MCQ)
Q: If f(x)=2x, what is f(3)?
A. 4
B. 5
C. 6
D. 7
Answer: \\boxed{C}

Example 2 (Free-form)
Q: Compute 2 + 3.
Solution: 2 + 3 = 5.
Final answer: \\boxed{5}
"""

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    examples_text = EXAMPLES
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        user = examples_text + "\n\n" + f"{question}\n\nOptions:\n{opts_text}"
        return SYSTEM_PROMPT_MCQ, user
    user = examples_text + "\n\n" + question
    return SYSTEM_PROMPT_MATH, user


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
Example 1 (MCQ)
Q: If f(x)=2x, what is f(3)?
A. 4
B. 5
C. 6
D. 7
Answer: \boxed{C}

Example 2 (Free-form)
Q: Compute 2 + 3.
Solution: 2 + 3 = 5.
Final answer: \boxed{5}


$int_{-infty}^{+infty} frac{a ...

── Free-form user prompt (first 200 chars) ──
Example 1 (MCQ)
Q: If f(x)=2x, what is f(3)?
A. 4
B. 5
C. 6
D. 7
Answer: \boxed{C}

Example 2 (Free-form)
Q: Compute 2 + 3.
Solution: 2 + 3 = 5.
Final answer: \boxed{5}


Find the sum of the first $32 ...



In [13]:
# # Force update quantization to match CUDA version dependencies.
# # 1. Remove conflicting older versions if they are lingering
# !pip uninstall -y nvidia-nvjitlink-cu12

# # 2. Install the correct, unified package name
# !pip install nvidia-nvjitlink

# # 3. Inject the path straight into your environment so bitsandbytes can see it
# import os
# import glob

# # Find where pip dropped the CUDA 13 JIT library file
# found_paths = glob.glob("/usr/local/lib/python3.*/dist-packages/nvidia/**/libnvJitLink.so.13", recursive=True)

# if found_paths:
#     cuda_dir = os.path.dirname(found_paths[0])
#     os.environ["LD_LIBRARY_PATH"] = f"{cuda_dir}:{os.environ.get('LD_LIBRARY_PATH', '')}"
#     print(f"✅ Successfully linked CUDA 13 JIT path: {cuda_dir}")
# else:
#     print("⚠️ Package installed, but the library file wasn't found in the standard path. Try restarting your session.")

✅ Successfully linked CUDA 13 JIT path: /usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib


In [15]:
from transformers import AutoTokenizer

# Initialize tokenizer before training functions use it
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
print(f"Tokenizer loaded for {MODEL_ID}")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Tokenizer loaded for Qwen/Qwen3-4B-Thinking-2507


In [16]:
import math
import torch
from torch.utils.data import DataLoader, Dataset
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, get_cosine_schedule_with_warmup

TRAIN_MAX_LENGTH = 4096
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 4
MAX_EPOCHS = 3
EARLY_STOPPING_PATIENCE = 2
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05

if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    TRAIN_DTYPE = torch.bfloat16
else:
    TRAIN_DTYPE = torch.float16


def _normalize_answer(answer):
    if isinstance(answer, list):
        answer = answer[0]
    return str(answer).strip()


def build_supervised_example(item):
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )

    answer_text = _normalize_answer(item["answer"])
    if item.get("options"):
        answer_text = answer_text.upper()
    target_text = f"\\boxed{{{answer_text}}}"
    full_text = prompt_text + target_text + tokenizer.eos_token
    return prompt_text, full_text


class SupervisedMathDataset(Dataset):
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        prompt_text, full_text = build_supervised_example(self.records[index])
        prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
        encoded = tokenizer(
            full_text,
            truncation=True,
            max_length=TRAIN_MAX_LENGTH,
            add_special_tokens=False,
        )
        labels = encoded["input_ids"].copy()
        prompt_length = min(len(prompt_ids), len(labels))
        labels[:prompt_length] = [-100] * prompt_length
        return {
            "input_ids": encoded["input_ids"],
            "attention_mask": encoded["attention_mask"],
            "labels": labels,
        }


def supervised_collate(batch):
    max_length = max(len(sample["input_ids"]) for sample in batch)
    input_ids = []
    attention_mask = []
    labels = []
    for sample in batch:
        pad_length = max_length - len(sample["input_ids"])
        input_ids.append(sample["input_ids"] + [tokenizer.pad_token_id] * pad_length)
        attention_mask.append(sample["attention_mask"] + [0] * pad_length)
        labels.append(sample["labels"] + [-100] * pad_length)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }


def compute_sequence_loss(logits, labels):
    loss_fn = torch.nn.CrossEntropyLoss(ignore_index=-100)
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()
    return loss_fn(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))


def evaluate_model(model, loader):
    model.eval()
    running_loss = 0.0
    running_batches = 0
    model_device = model.get_input_embeddings().weight.device
    with torch.no_grad():
        for batch in loader:
            batch = {key: value.to(model_device) for key, value in batch.items()}
            with torch.autocast(device_type="cuda", dtype=TRAIN_DTYPE, enabled=torch.cuda.is_available()):
                outputs = model(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    use_cache=False,
                )
                loss = compute_sequence_loss(outputs.logits, batch["labels"])
            running_loss += loss.item()
            running_batches += 1
    return running_loss / max(1, running_batches)


def train_fold(val_fold_id):
    print(f"\n─── Training Fold (val={val_fold_id}) ───")
    train_indices = [idx for idx in trainval_indices if fold_assignments[idx] != val_fold_id]
    val_indices = [idx for idx in trainval_indices if fold_assignments[idx] == val_fold_id]
    train_data = _select_records(train_indices)
    val_data = _select_records(val_indices)

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=TRAIN_DTYPE,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    train_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        quantization_config=bnb_config,
        device_map="auto",
    )
    train_model = prepare_model_for_kbit_training(train_model)
    train_model.config.use_cache = False
    train_model.gradient_checkpointing_enable()

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )
    train_model = get_peft_model(train_model, lora_config)

    train_loader = DataLoader(
        SupervisedMathDataset(train_data),
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True,
        collate_fn=supervised_collate,
    )
    val_loader = DataLoader(
        SupervisedMathDataset(val_data),
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        collate_fn=supervised_collate,
    )

    optimizer = torch.optim.AdamW(train_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    updates_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
    total_updates = max(1, updates_per_epoch * MAX_EPOCHS)
    warmup_steps = max(1, int(total_updates * WARMUP_RATIO))
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_updates)
    model_device = train_model.get_input_embeddings().weight.device

    best_val_loss = float("inf")
    patience = 0
    for epoch in range(MAX_EPOCHS):
        train_model.train()
        optimizer.zero_grad(set_to_none=True)
        running_train_loss = 0.0

        for step, batch in enumerate(train_loader, start=1):
            batch = {key: value.to(model_device) for key, value in batch.items()}
            with torch.autocast(device_type="cuda", dtype=TRAIN_DTYPE, enabled=torch.cuda.is_available()):
                outputs = train_model(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    use_cache=False,
                )
                loss = compute_sequence_loss(outputs.logits, batch["labels"])
                loss = loss / GRAD_ACCUM_STEPS

            loss.backward()
            running_train_loss += loss.item() * GRAD_ACCUM_STEPS

            if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
                torch.nn.utils.clip_grad_norm_(train_model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

        val_loss = evaluate_model(train_model, val_loader)
        train_loss = running_train_loss / max(1, len(train_loader))
        print(f"Epoch {epoch + 1}/{MAX_EPOCHS} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience = 0
            adapter_dir = Path(f"artifacts/qwen3b_lora_fold_{val_fold_id}")
            adapter_dir.mkdir(parents=True, exist_ok=True)
            train_model.save_pretrained(adapter_dir)
            tokenizer.save_pretrained(adapter_dir)
            print(f"Saved best adapter to {adapter_dir}")
        else:
            patience += 1
            if patience >= EARLY_STOPPING_PATIENCE:
                print("Early stopping triggered.")
                break

    del train_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return best_val_loss


all_fold_losses = []
for fold_id in range(N_FOLDS):
    loss = train_fold(fold_id)
    all_fold_losses.append(loss)

print(f"\n─── K-Fold Summary ───")
for i, loss in enumerate(all_fold_losses):
    print(f"Fold {i} best val_loss: {loss:.4f}")
avg_loss = sum(all_fold_losses) / len(all_fold_losses)
print(f"Average best val_loss: {avg_loss:.4f}")


─── Training Fold (val=0) ───


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


KeyboardInterrupt: 

In [ ]:
import sys
import io

# Patch the Jupyter stdout so fileno() doesn't crash
class PatchedStdout:
    def __init__(self, original):
        self._original = original
    def fileno(self):
        return 1  # fake a real file descriptor
    def __getattr__(self, name):
        return getattr(self._original, name)

sys.stdout = PatchedStdout(sys.stdout)

# Now load vLLM
from vllm import LLM

In [ ]:
!pip install bitsandbytes==0.45.5 peft --upgrade -q
!pip install triton -q

## 5. Load Model with vLLM (for general case, vLLM is faster)

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
llm = LLM(
    model=MODEL_ID,
    dtype="float16",
    gpu_memory_utilization=0.50,
    max_model_len=16384,
    trust_remote_code=True,
    max_num_seqs=1,
    max_num_batched_tokens=32768,
    enable_prefix_caching=False,
)

sampling_params = SamplingParams(
     max_tokens=MAX_TOKENS,
     temperature=0.7,
     top_p=0.95,
     top_k=20,
     min_p=0.0,
     presence_penalty=0.0,
     repetition_penalty=1.0,
 )

print("Model loaded.")

## 5. Load Model with Transformers (alternative to vLLM for DataHub)

We load **Qwen3-4B-Thinking-2507** with **INT4 quantization** via BitsAndBytes.  

Key parameters:
- `load_in_4bit` — quantization strategy of INT4

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
)

# Find the best fold
best_fold_id = -1
best_fold_loss = float("inf")
for i, loss in enumerate(all_fold_losses):
    if loss < best_fold_loss:
        best_fold_loss = loss
        best_fold_id = i

BEST_ADAPTER_DIR = Path(f"artifacts/qwen3b_lora_fold_{best_fold_id}")
adapter_ready = (BEST_ADAPTER_DIR / "adapter_config.json").exists()

if adapter_ready:
    llm = PeftModel.from_pretrained(base_model, BEST_ADAPTER_DIR)
    print(f"Loaded best adapter from fold {best_fold_id} ({BEST_ADAPTER_DIR})")
else:
    llm = base_model
    print("Loaded base model without a fine-tuned adapter.")

llm.eval()

## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

### Generate with vLLM

In [ ]:
# Build prompts for first 5 entries
prompts = []
for item in test_data[:10]:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate
print(f"Generating responses for {len(prompts)} questions...")
outputs = llm.generate(prompts, sampling_params=sampling_params)

responses = [out.outputs[0].text.strip() for out in outputs]

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={test_data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

### Generate with Transformers (for Datahub)

In [ ]:
# Build prompts for first 5 entries
prompts = []
for item in test_data[:5]:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Tokenize (padded batch)
print(f"Generating responses for {len(prompts)} questions...")
model_device = next(llm.parameters()).device
inputs = tokenizer(
    prompts,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=16384,
).to(model_device)

# Generate
with torch.no_grad():
    output_ids = llm.generate(
        **inputs,
        max_new_tokens=MAX_TOKENS,
        temperature=0.6,
        top_p=0.95,
        top_k=20,
        repetition_penalty=1.0,
        do_sample=True,
    )

# Decode only the new tokens (strip the prompt)
responses = []
for i, out in enumerate(output_ids):
    new_tokens = out[inputs["input_ids"].shape[1]:]
    responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={test_data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

## 7. Score Responses

Scoring now runs on the held-out test split only.

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [ ]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(test_data, responses), total=len(test_data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

## 8. Summary

Print accuracy broken down by question type.

In [ ]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("TEST EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

## 9. Save Results

Results are written as newline-delimited JSON for the held-out test split.

**With evaluation**: each line is `{id, is_mcq, gold, response, correct}`.

**Without evaluation**: each line is `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [ ]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!